In [1]:
import torch
import torch.nn as nn

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [3]:
import torch
print(torch.__version__)
dataset_path = "../data/raw"

2.11.0+cu128


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [5]:
full_dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=train_transform
)

print("Classes:", len(full_dataset.classes))
print("Images:", len(full_dataset))

Classes: 15
Images: 20638


In [6]:
from torch.utils.data import random_split

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size]
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Train: 16510
Test: 4128


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [8]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [9]:
model = models.resnet18(weights="DEFAULT")

model.fc = nn.Linear(
    in_features=512,
    out_features=15
)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=15, bias=True)


In [10]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(total_params)

11184207


In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [12]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print(outputs.shape)

torch.Size([32, 15])


In [13]:
print(model.fc)
total_params
outputs.shape

Linear(in_features=512, out_features=15, bias=True)


torch.Size([32, 15])

In [14]:
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} Average Loss : {avg_loss}"
    )

Epoch 1 Average Loss : 0.40588536805855907
Epoch 2 Average Loss : 0.19109307495559485
Epoch 3 Average Loss : 0.1483856235665103
Epoch 4 Average Loss : 0.12895882123825864
Epoch 5 Average Loss : 0.11801084038817736
Epoch 6 Average Loss : 0.09669687861013551
Epoch 7 Average Loss : 0.08118064010262223
Epoch 8 Average Loss : 0.09434394213894848
Epoch 9 Average Loss : 0.06768293739923695
Epoch 10 Average Loss : 0.07939598103691896


In [15]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs,dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = (correct / total) * 100
print("Test Accuracy :", accuracy)

Test Accuracy : 95.32461240310077


In [16]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

In [17]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)
for i in range(len(cm)):
    accuracy = cm[i][i]/cm[i].sum()*100
    print("Class",i,"Accuracy :",accuracy,"%")

Class 0 Accuracy : 99.47089947089947 %
Class 1 Accuracy : 99.36102236421725 %
Class 2 Accuracy : 96.65071770334929 %
Class 3 Accuracy : 100.0 %
Class 4 Accuracy : 92.5925925925926 %
Class 5 Accuracy : 97.14964370546319 %
Class 6 Accuracy : 74.27184466019418 %
Class 7 Accuracy : 94.50261780104712 %
Class 8 Accuracy : 98.84393063583815 %
Class 9 Accuracy : 96.88473520249221 %
Class 10 Accuracy : 84.98583569405099 %
Class 11 Accuracy : 98.93992932862191 %
Class 12 Accuracy : 98.56687898089172 %
Class 13 Accuracy : 95.55555555555556 %
Class 14 Accuracy : 96.62576687116564 %


In [18]:
torch.save(
    model.state_dict(),
    "model.pth"
)